## studentRegistration(수강신청) 데이터 체크(1차 전처리)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

# 노트북에서 실행할 경우 프로젝트 최상위 폴더로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

STUDENT_REGISTRATION_PATH = PROJECT_ROOT / "data" / "raw" / "studentRegistration.csv"

print("프로젝트 위치:", PROJECT_ROOT)
print("studentRegistration 존재:", STUDENT_REGISTRATION_PATH.exists())

프로젝트 위치: C:\SKN_AI\oulad-churn-prediction
studentRegistration 존재: True


In [2]:
student_registration = pd.read_csv(STUDENT_REGISTRATION_PATH)

print("student_registration 크기:", student_registration.shape)
display(student_registration.head())

student_registration 크기: (32593, 5)


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159,?
1,AAA,2013J,28400,-53,?
2,AAA,2013J,30268,-92,12
3,AAA,2013J,31604,-52,?
4,AAA,2013J,32885,-176,?


In [3]:
print("===== student_registration 정보 =====")
student_registration.info()

print("\nstudent_registration 컬럼별 결측/공백/'?' 개수:")
for col in student_registration.columns:
    none_count = (
        student_registration[col].isnull()
        | (student_registration[col].astype(str).str.strip() == "")
        | (student_registration[col].astype(str).values == "?")
    ).sum()
    if none_count > 0:
        print(f"{col} : {none_count}")

===== student_registration 정보 =====
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   code_module          32593 non-null  str  
 1   code_presentation    32593 non-null  str  
 2   id_student           32593 non-null  int64
 3   date_registration    32593 non-null  str  
 4   date_unregistration  32593 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.6 MB

student_registration 컬럼별 결측/공백/'?' 개수:
date_registration : 45
date_unregistration : 22521


- `date_registration` 결측 45건: 등록일이 기록되지 않은 실제 결측치 → 해당 행 제거
- `date_unregistration` 결측 22,521건: 강좌를 끝까지 수강해 **중도 철회하지 않은** 정상 데이터 → 값을 채우지 않고 결측 그대로 유지


In [4]:
# 수강신청일 결측치인 row 제거
student_registration_processed = student_registration[
    student_registration["date_registration"].astype(str).str.strip() != "?"
].copy()

# 정수형으로 변환
student_registration_processed["date_registration"] = (
    student_registration_processed["date_registration"].astype(int)
)

# '?' 문자열을 결측치로 바꾸고 실수형으로 변환 (철회하지 않은 학생은 NaN 유지)
student_registration_processed["date_unregistration"] = (
    student_registration_processed["date_unregistration"]
    .replace("?", np.nan)
    .astype(float)
)

print("제거 전 행 수:", len(student_registration))
print("제거 후 행 수:", len(student_registration_processed))
print("\ndate_registration dtype:", student_registration_processed["date_registration"].dtype)
print("date_unregistration dtype:", student_registration_processed["date_unregistration"].dtype)
print(
    "date_unregistration 결측 행 수:",
    student_registration_processed["date_unregistration"].isna().sum()
)

제거 전 행 수: 32593
제거 후 행 수: 32548

date_registration dtype: int64
date_unregistration dtype: float64
date_unregistration 결측 행 수: 22515


## 컬럼 추가


In [5]:
def check_unregister(value):
    return "Y" if pd.isna(value) else "N"


def count_unregi_week(value):
    if pd.isna(value):
        return "N"
    week = (float(value) // 7 + 1) if (float(value) % 7 > 0) else float(value) // 7
    return str(week)


# 수강철회 여부 컬럼 추가
student_registration_processed["unregister_yn"] = (
    student_registration_processed["date_unregistration"].apply(check_unregister)
)

# 이탈 주차 컬럼 추가 (철회하지 않은 학생은 'N')
student_registration_processed["unregister_week"] = (
    student_registration_processed["date_unregistration"].apply(count_unregi_week)
)

print("unregister_yn 분포:")
print(student_registration_processed["unregister_yn"].value_counts())

print("\nunregister_week 상위 값:")
print(student_registration_processed["unregister_week"].value_counts().head(10))

unregister_yn 분포:
unregister_yn
Y    22515
N    10033
Name: count, dtype: int64

unregister_week 상위 값:
unregister_week
N       22515
2.0      1057
0.0       897
-1.0      456
4.0       370
1.0       317
-2.0      303
5.0       294
8.0       254
7.0       248
Name: count, dtype: int64


기존 노트북에서는 `date_unregistration`이 이미 실수형으로 변환된 뒤 `x is None`, `x == '?'` 조건으로 철회 여부를 판단했다.
변환된 결측치는 `NaN`(float)이라 두 조건 모두 만족하지 못해 `unregister_yn`이 항상 `'N'`으로만 나오는 문제가 있었다.
이번 버전은 `pd.isna()`로 판단하도록 수정했다. 이탈 예측 프로젝트의 핵심 라벨과 직결되는 부분이라 아래에서 결과를 한 번 더 검증한다.


In [6]:
# date_unregistration 결측 여부와 unregister_yn이 서로 일치하는지 검증
expected_unregister_yn = np.where(
    student_registration_processed["date_unregistration"].isna(),
    "Y",
    "N"
)

mismatch_count = (
    student_registration_processed["unregister_yn"].values
    != expected_unregister_yn
).sum()

print("unregister_yn 불일치 행 수:", mismatch_count)

STUDENT_REGISTRATION_KEYS = ["code_module", "code_presentation", "id_student"]

print(
    "\n키 중복 행 수:",
    student_registration_processed.duplicated(subset=STUDENT_REGISTRATION_KEYS).sum()
)

display(student_registration_processed.head())

unregister_yn 불일치 행 수: 0

키 중복 행 수: 0


,code_module,code_presentation,id_student,date_registration,date_unregistration,unregister_yn,unregister_week
0,AAA,2013J,11391,-159,NaN,Y,N
1,AAA,2013J,28400,-53,NaN,Y,N
2,AAA,2013J,30268,-92,12.0,N,2.0
3,AAA,2013J,31604,-52,NaN,Y,N
4,AAA,2013J,32885,-176,NaN,Y,N


In [7]:
# 저장 경로
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_REGISTRATION_OUTPUT_PATH = INTERIM_DIR / "student_registration_processed.csv"

student_registration_processed.to_csv(STUDENT_REGISTRATION_OUTPUT_PATH, index=False)

print("저장 완료")
print(
    STUDENT_REGISTRATION_OUTPUT_PATH.name,
    f"{STUDENT_REGISTRATION_OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB"
)

저장 완료
student_registration_processed.csv 0.92 MB
